## PX1 SD — третий этап: содержимое Linux-раздела

Продолжение после проверки `linux_ltib`. Этот блокнот запускается **на Google Colab, а не на телефоне**. Исходный `ЛП.imgc` только читается. Временная копия раздела (~7.9 ГБ) создаётся в Colab, изучается в режиме чтения и удаляется. Никакие исполняемые файлы из образа не запускаются. Результат `PX1_SD_OS_REPORT.txt` сохраняется в личный Google Диск.

Блокнот анализирует **только** сведения об ОС, архитектуру и выбранные системные каталоги; не читает ключи, пароли, пользовательские файлы или записи инспекций. Убедись, что доверяешь подключению Colab к Google Диску перед подтверждением доступа.

Нажми **Среда выполнения → Выполнить все**. После завершения сообщи ChatGPT: «Третий отчёт готов».


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, struct, shutil
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive')
src = root / 'Untitled folder' / 'ЛП.imgc'
if not src.is_file():
    src = next((Path(d)/'ЛП.imgc' for d,_,names in os.walk(root) if 'ЛП.imgc' in names), None)
if not src or not src.is_file(): raise FileNotFoundError('Не найден ЛП.imgc')
project=Path('/content/unimgc')
if not project.is_dir():
    subprocess.run(['git','clone','--depth','1','https://github.com/Shizmob/unimgc.git',str(project)],check=True)
subprocess.run(['make','-C',str(project)],check=True)
part=Path('/content/px1_ro_partition.img')
offset=32901120
size=7912719360
if shutil.disk_usage('/content').free < size + 1024**3:
    raise RuntimeError('Мало свободного места в Colab для копии раздела')
print('Читаю раздел в временный файл Colab (примерно 7.9 ГБ)...')
p=subprocess.Popen([str(project/'unimgc'),str(src)],stdout=subprocess.PIPE,stderr=subprocess.DEVNULL)
try:
    toskip=offset
    while toskip:
        data=p.stdout.read(min(4*1024*1024,toskip))
        if not data: raise EOFError('Образ закончился до начала раздела')
        toskip-=len(data)
    left=size
    with part.open('wb') as out:
        while left:
            data=p.stdout.read(min(4*1024*1024,left))
            if not data: raise EOFError('Образ закончился внутри раздела')
            out.write(data)
            left-=len(data)
finally:
    p.stdout.close()
    p.terminate()
    try: p.wait(timeout=10)
    except subprocess.TimeoutExpired: p.kill(); p.wait()
def dbg(command):
    v=subprocess.run(['debugfs','-R',command,str(part)],capture_output=True,timeout=90)
    return v.stdout.decode('utf-8','replace')
report=['PX1 Mini-Cam — офлайн-анализ системной информации Linux-раздела', 'Метка раздела по предыдущей проверке: linux_ltib']
for folder in ['/','/boot','/lib/modules','/etc','/opt','/usr/local']:
    raw=dbg('ls -p '+folder)
    names=[]
    for item in raw.splitlines():
        parts=item.strip().split('/')
        if len(parts)>=6 and parts[5]: names.append(parts[5])
    report.append('Каталог '+folder+' (до 35 имён): '+(', '.join(names[:35]) if names else 'нет данных'))
for f in ['/etc/os-release','/usr/lib/os-release','/etc/issue','/etc/ltib-release','/etc/debian_version']:
    raw=dbg('cat '+f)
    if raw and len(raw)<12000 and all(c.isprintable() or c in '\n\r\t' for c in raw):
        report.append(f+' (до 15 строк):\n'+'\n'.join(raw.splitlines()[:15]))
arch={3:'x86',40:'ARM',62:'x86-64',183:'AArch64',8:'MIPS',20:'PowerPC',4:'m68k'}
for f in ['/bin/busybox','/sbin/init','/bin/ls','/usr/bin/env']:
    dst=Path('/content/px1_elf_probe')
    if dst.exists(): dst.unlink()
    dbg('dump '+f+' '+str(dst))
    if dst.is_file():
        header=dst.open('rb').read(64)
        dst.unlink()
        if header[:4]==b'\x7fELF' and header[5] in (1,2):
            order='<' if header[5]==1 else '>'
            machine=struct.unpack_from(order+'H',header,18)[0]
            report.append('Архитектура ELF '+f+': '+arch.get(machine,'код '+str(machine)))
            break
report.append('Подтверждённых уязвимостей этот отчёт не устанавливает.')
out=root/'PX1_SD_OS_REPORT.txt'
out.write_text('\n'.join(report)+'\n',encoding='utf-8')
part.unlink(missing_ok=True)
print('\n'.join(report))
print('Готово! Отчёт сохранён на Google Диске:',out.name)
